# Join GDP data to Forest Area table

## What this does
Adds a `GDP per capita (current US$)` column to the forest area dataset by:
1. **Unpivoting** the GDP table from wide format (years as columns) to long format (one row per country per year) using pandas `melt`
2. **Joining** via SQL (DuckDB) on `country code` + `year` — avoids name-mismatch issues entirely
3. **Saving** the result to a new CSV — originals are not modified

## Key decisions
- **Join key**: `forest.Code = gdp.Country Code` + `forest.Year = gdp.year` — 3-letter ISO codes are consistent across both files, more reliable than matching on country name
- **Join type**: LEFT JOIN — all forest rows are kept; GDP is NULL where no match exists
- **Engine**: DuckDB — fast SQL engine that runs locally with no server

## Rows that will have NULL GDP
| Reason | Example |
|--------|---------|
| Year outside 1960–2024 | Historical rows (pre-1960), year 2025 |
| Diff rows | `Year = 'diff_2025_minus_1990'` |
| Regional aggregates (no ISO code) | European Union (27), High-income countries |

## Output
`data_raw/MAIN_1990-2025forest-area-gdp-joined.csv`

In [21]:
%pip install duckdb --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

FOREST = Path('../data_raw/MAIN_1990-2025forest-area-as-share-of-land-area.csv')
GDP    = Path('../data_dan/1960-2024_GDP_USD_clean.csv')
OUT    = Path('../data_dan/MAIN_Forest_GDP_joined.csv')

con = duckdb.connect()

In [23]:
# Step 1: Unpivot GDP from wide to long using pandas melt
# Wide format: one row per country, year values spread across columns ("1960", "1961", ...)
# Long format: one row per country per year
gdp_wide = pd.read_csv(GDP)
year_cols = [c for c in gdp_wide.columns if c.isdigit() and len(c) == 4]

gdp_long = gdp_wide.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=year_cols,
    var_name='year',
    value_name='GDP per capita (current US$)'
)
gdp_long['year'] = gdp_long['year'].astype(int)

print(f'GDP long format: {len(gdp_long):,} rows')
gdp_long.head(3)

GDP long format: 17,290 rows


,Country Name,Country Code,year,GDP per capita (current US$)
0,Aruba,ABW,1960,NaN
1,Africa Eastern and Southern,AFE,1960,186.089204
2,Afghanistan,AFG,1960,NaN


In [24]:
# Step 2: Register both tables in DuckDB and run the SQL join
forest = pd.read_csv(FOREST)
con.register('forest', forest)
con.register('gdp_long', gdp_long)

result = con.execute('''
    SELECT
        f.Entity,
        f.Code,
        f.Year,
        f."Share of land covered by forest",
        f."Share of land covered by forest (Annotations)",
        g."GDP per capita (current US$)"
    FROM forest f
    LEFT JOIN gdp_long g
        ON  f.Code = g."Country Code"
        AND TRY_CAST(f.Year AS INTEGER) = g.year
''').df()

print(f'Total rows:    {len(result):,}')
print(f'GDP matched:   {result["GDP per capita (current US$)"].notna().sum():,}')
print(f'NULL GDP:      {result["GDP per capita (current US$)"].isna().sum():,}')
result.head(5)

Total rows:    7,970
GDP matched:   7,041
NULL GDP:      929


,Entity,Code,Year,Share of land covered by forest,Share of land covered by forest (Annotations),GDP per capita (current US$)
0,Japan,JPN,1985,65.600000,NaN,11809.460345
1,Vietnam,VNM,1985,30.000000,NaN,238.647805
2,United States,USA,1987,32.380000,NaN,20038.941099
3,Aruba,ABW,1990,2.722222,NaN,12187.536361
4,Afghanistan,AFG,1990,1.854315,NaN,NaN


In [25]:
# Step 3: Sort by country name then by year, save to new CSV
result.sort_values(['Entity', 'Year'], inplace=True)
result.to_csv(OUT, index=False)
print(f'Saved to {OUT}')

Saved to ..\data_dan\MAIN_Forest_GDP_joined.csv
